# Election Polls + Results → Long Modeling Dataset

**Goal (this stage):** keep as much data as possible. Output is **one row per individual poll**, with the race's eventual result attached. No collapsing yet — that comes later.

So the table is one-to-many: many poll rows per race, each carrying who won and the final result.

**Scope:** Senate, House, Governor · 2000–present · U.S.

**Sources (both public, free):**
- **Polls:** FiveThirtyEight raw poll files (`*_polls.csv`) — current + historical
- **Results (who won):** FiveThirtyEight `election-results` repo — has a per-candidate `winner` flag

Run top to bottom. First run downloads into a local `data/` folder and caches it.

In [1]:
import os, io, re, unicodedata
import pandas as pd
import numpy as np
import requests

pd.set_option('display.max_columns', 80)
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

MIN_YEAR = 2000          # results go back to 1998; polling thins out pre-2000
OFFICES  = ['Senate', 'House', 'Governor']
print('pandas', pd.__version__)

pandas 3.0.1


## 1. Download poll + result data

FiveThirtyEight was dissolved (Mar 2025) and its poll endpoints now return HTML, so we use live replacements that share the **same 538 schema**:

- **Current polls (latest cycle):** New York Times poll CSVs (`nytimes.com/newsgraphics/polls/*.csv`) — same columns as old 538, updated continuously, CC-BY.
- **Historical polls (back to ~2000):** Internet Archive snapshot of 538's `*_polls_historical.csv` (frozen but complete).
- **Results / winners:** FiveThirtyEight `election-results` GitHub repo (still live).

All cached in `data/`. The Archive can rate-limit (HTTP 429) — just re-run the cell; cached files are skipped.

In [ ]:
def fetch(name, urls, force=False, min_kb=20):
    """Return data/<name>, using the local copy if present (no download), else download.

    The frozen historical poll snapshots are COMMITTED to the repo (see .gitignore), so on a
    fresh clone they already exist here and this never touches the network for them. Only the
    current-cycle NYT files and result files are fetched live. Rejects HTML/tiny error pages.
    """
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path) and os.path.getsize(path) > min_kb * 1024 and not force:
        print(f'cached  {name}  ({os.path.getsize(path)//1024} KB)')
        return path
    last = None
    for u in urls:
        try:
            r = requests.get(u, timeout=90,
                             headers={'User-Agent': 'Mozilla/5.0 (research)',
                                      'Accept': 'text/csv,text/plain,*/*'})
            r.raise_for_status()
            head = r.content[:300].lstrip().lower()
            if head.startswith((b'<!doctype', b'<html', b'<')):
                raise ValueError('got HTML, not CSV (endpoint likely dead/rate-limited)')
            if len(r.content) < min_kb * 1024:
                raise ValueError(f'file too small ({len(r.content)} bytes) - likely an error page')
            with open(path, 'wb') as f:
                f.write(r.content)
            print(f'OK      {name}  ({len(r.content)//1024} KB)  <- {u[:70]}')
            return path
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            print(f'  skip  {u[:70]}  ({last})')
    print(f'  !! all mirrors failed for {name}: {last}')
    return None

# ---- CURRENT polls: New York Times (538 schema) — these DO refresh, so always fetched live ----
NYT = 'https://www.nytimes.com/newsgraphics/polls/'

# ---- HISTORICAL polls: frozen Internet Archive snapshots of 538 ----
# These never change and are COMMITTED to data/ (*_historical.csv). The Wayback URL below is only a
# fallback if the committed copy is somehow missing — the Archive is slow/rate-limited, so we avoid it.
def wb(path):
    return f'http://web.archive.org/web/2id_/https://projects.fivethirtyeight.com/polls/data/{path}'

poll_sources = {
    'Senate':   {'current': NYT + 'senate.csv',   'historical': wb('senate_polls_historical.csv')},
    'House':    {'current': NYT + 'house.csv',     'historical': wb('house_polls_historical.csv')},
    'Governor': {'current': NYT + 'governor.csv',  'historical': wb('governor_polls_historical.csv')},
}

poll_files = {}
for office, src in poll_sources.items():
    cur  = fetch(f'{office.lower()}_current.csv',    [src['current']])
    hist = fetch(f'{office.lower()}_historical.csv', [src['historical']])   # committed; no network on fresh clone
    poll_files[office] = [p for p in (cur, hist) if p]

# ---- RESULTS: FiveThirtyEight election-results repo (live GitHub, reliable) ----
def res_urls(fn):
    return [
        f'https://raw.githubusercontent.com/fivethirtyeight/election-results/main/{fn}',
        f'https://cdn.jsdelivr.net/gh/fivethirtyeight/election-results@main/{fn}',
    ]
result_files = {
    'Senate':   fetch('res_senate.csv',   res_urls('election_results_senate.csv')),
    'House':    fetch('res_house.csv',    res_urls('election_results_house.csv')),
    'Governor': fetch('res_governor.csv', res_urls('election_results_gubernatorial.csv')),
}

## 2. Inspect raw columns

Schemas drift over time. Run once and eyeball the columns; if the standardizing step below references a column that's missing, adjust the `pick()` calls.

In [3]:
for office in OFFICES:
    print(f'=== {office} POLL FILES ===')
    for f in poll_files.get(office, []):
        df = pd.read_csv(f, nrows=3, low_memory=False)
        print(f'  {os.path.basename(f)}  ({df.shape[1]} cols): {list(df.columns)}')
    if result_files.get(office):
        r = pd.read_csv(result_files[office], nrows=3, low_memory=False)
        print(f'  RESULTS ({r.shape[1]} cols): {list(r.columns)}')
    print()

=== Senate POLL FILES ===
  senate_current.csv  (48 cols): ['poll_id', 'pollster_id', 'pollster', 'sponsor_ids', 'sponsors', 'display_name', 'pollster_rating_id', 'pollster_rating_name', 'numeric_grade', 'pollscore', 'methodology', 'transparency_score', 'state', 'start_date', 'end_date', 'sponsor_candidate_id', 'sponsor_candidate', 'sponsor_candidate_party', 'question_id', 'sample_size', 'population', 'subpopulation', 'population_full', 'tracking', 'created_at', 'notes', 'url', 'url_article', 'url_topline', 'url_crosstab', 'source', 'internal', 'partisan', 'cycle', 'office_type', 'seat_name', 'seat_number', 'election_date', 'stage', 'party', 'pct', 'answer', 'candidate_name', 'candidate_id', 'race_id', 'nationwide_match', 'ranked_choice_reallocated', 'hypothetical']
  senate_historical.csv  (52 cols): ['poll_id', 'pollster_id', 'pollster', 'sponsor_ids', 'sponsors', 'display_name', 'pollster_rating_id', 'pollster_rating_name', 'numeric_grade', 'pollscore', 'methodology', 'transparency_

## 3. Helpers + load polls (LONG: one row per poll-candidate)

We keep every poll row and a generous set of columns (pollster, dates, sample size, rating, methodology, etc.) so nothing is thrown away. Join keys are normalized but the original columns are preserved.

In [4]:
STATE_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','District of Columbia':'DC',
    'Florida':'FL','Georgia':'GA','Hawaii':'HI','Idaho':'ID','Illinois':'IL',
    'Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY','Louisiana':'LA',
    'Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR',
    'Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD',
    'Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA',
    'Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY',
}
# only the 50 states elect the offices we model; drop territories / nationwide rows
VALID_STATES = set(STATE_ABBR.values())

# Polls use full state names ('Arizona'); results use abbrevs ('AZ'). Standardize both to abbrev.
def to_abbr(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if len(s) == 2:
        return s.upper()
    return STATE_ABBR.get(s, s)

# Extract a House district number ('7', '12') from either a numeric seat_number (6.0)
# or a word/number seat_name ('District 7', 'Third Congressional District', 'At-Large').
_ORD = {'first':'1','second':'2','third':'3','fourth':'4','fifth':'5','sixth':'6',
        'seventh':'7','eighth':'8','ninth':'9','tenth':'10','eleventh':'11','twelfth':'12',
        'thirteenth':'13','fourteenth':'14','fifteenth':'15','sixteenth':'16','seventeenth':'17',
        'eighteenth':'18','nineteenth':'19','twentieth':'20','twenty-first':'21','twenty-second':'22'}
def parse_district(val):
    if pd.isna(val):
        return None
    s = str(val).lower()
    m = re.search(r'(\d+)', s)
    if m:
        return str(int(m.group(1)))
    if 'at-large' in s or 'at large' in s:
        return '1'
    for w, n in _ORD.items():
        if w in s:
            return n
    return None

# Placeholder / non-candidate poll answers that are not real people running.
JUNK_ANSWERS = {
    'generic democrat', 'generic republican', 'generic ballot',
    "don't know", 'dont know', 'undecided', 'someone else', 'other',
    'would not vote', 'neither', 'no opinion', 'refused', 'none of these',
    'skipped', 'will not vote', 'not sure',
}
def is_junk_answer(name):
    return str(name).strip().lower() in JUNK_ANSWERS

# Normalize a candidate name for joining: strip accents, lowercase -> 'lastname firstinitial'
def norm_name(s):
    if pd.isna(s):
        return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    s = re.sub(r'\b(jr|sr|ii|iii|iv)\b', '', s)
    s = re.sub(r'[^a-z\s]', ' ', s)
    parts = [w for w in s.split() if w]
    if not parts:
        return None
    last = parts[-1]
    fi = parts[0][0] if parts[0] != last else ''
    return f'{last} {fi}'.strip()

def norm_party(p):
    if pd.isna(p):
        return 'OTH'
    p = str(p).strip().upper()
    if p.startswith('DEM'): return 'DEM'
    if p.startswith('REP'): return 'REP'
    return 'OTH'

def pick(df, *cands):
    lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

In [5]:
def load_polls(paths, office):
    frames = []
    for path in paths:
        df = pd.read_csv(path, low_memory=False)
        cols = {
            'year':       pick(df, 'cycle', 'election_year', 'year'),
            'state':      pick(df, 'state'),
            'candidate':  pick(df, 'candidate_name', 'answer', 'candidate'),
            'poll_party': pick(df, 'party'),
            'pct':        pick(df, 'pct'),
            'end_date':   pick(df, 'end_date', 'enddate'),
        }
        # district sources (House): numeric seat_number, with word/number seat_name as fallback
        seat_num  = pick(df, 'seat_number')
        seat_name = pick(df, 'seat_name')

        missing = [k for k, v in cols.items() if v is None]
        if missing:
            print(f'  !! skipping {os.path.basename(path)} ({office}) - missing {missing}')
            continue

        rename_map = {src: tgt for tgt, src in cols.items() if src != tgt}
        targets = set(cols.keys())
        collisions = [c for c in df.columns
                      if c in targets and c not in rename_map.keys() and c not in cols.values()]
        if collisions:
            df = df.drop(columns=collisions)
        df = df.rename(columns=rename_map)
        df = df.loc[:, ~df.columns.duplicated()]

        df['office']   = office
        df['year']     = pd.to_numeric(df['year'], errors='coerce')
        df['pct']      = pd.to_numeric(df['pct'], errors='coerce')
        df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce', format='mixed')
        df['state']    = df['state'].map(to_abbr)
        # district (House only): seat_number first, fall back to seat_name word/number form
        if office == 'House':
            di = df[seat_num].map(parse_district) if seat_num else pd.Series(index=df.index, dtype=object)
            if seat_name:
                di = di.where(di.notna(), df[seat_name].map(parse_district))
            df['district'] = di
        else:
            df['district'] = ''
        df['cand_key']  = df['candidate'].map(norm_name)
        df['party_std'] = df['poll_party'].map(norm_party)

        # --- drop non-races: placeholder answers, nationwide/generic-ballot, territories ---
        df = df[~df['candidate'].map(is_junk_answer)]
        df = df[df['state'].isin(VALID_STATES)]          # removes 'US' generic ballot + PR/territories
        if office == 'House':
            df = df[df['district'].notna()]              # a House poll without a district can't be placed

        df = df.dropna(subset=['year', 'pct', 'cand_key'])
        df['year'] = df['year'].astype(int)
        frames.append(df)

    if not frames:
        raise RuntimeError(f'No usable poll files for {office}')
    out = pd.concat(frames, ignore_index=True)
    dedup_cols = [c for c in ['poll_id','question_id','candidate','pct','end_date'] if c in out.columns]
    if dedup_cols:
        out = out.drop_duplicates(subset=dedup_cols)
    return out

polls = pd.concat([load_polls(poll_files[o], o) for o in OFFICES if poll_files.get(o)],
                  ignore_index=True)
polls['district'] = polls['district'].fillna('')
polls = polls[polls['year'] >= MIN_YEAR]
print('total poll rows (long):', len(polls))
print('year range:', polls['year'].min(), '-', polls['year'].max())
print('by office:\n', polls.groupby('office').size())
print('house districts seen:', sorted(polls[polls.office=="House"]["district"].unique())[:15], '...')
polls[['year','state','office','district','candidate','party_std','pct','end_date','cand_key']].head()

total poll rows (long): 22546
year range: 2018 - 2026
by office:
 office
Governor     7474
House        4817
Senate      10255
dtype: int64
house districts seen: ['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22'] ...


,year,state,office,district,candidate,party_std,pct,end_date,cand_key
0,2026,ME,Senate,,Graham Platner,DEM,47.6,2026-06-14,platner g
1,2026,ME,Senate,,Susan M. Collins,REP,45.4,2026-06-14,collins s
2,2026,GA,Senate,,Derek Dooley,REP,48.0,2026-06-15,dooley d
3,2026,GA,Senate,,Mike Collins,REP,50.0,2026-06-15,collins m
4,2026,MA,Senate,,John Deaton,REP,26.2,2026-06-12,deaton j


## 4. Load results (the outcome per candidate per race)

In [6]:
def load_results(path, office):
    df = pd.read_csv(path, low_memory=False)
    sc = pick(df, 'stage')
    if sc:
        df = df[df[sc].astype(str).str.lower().str.contains('general', na=False)]
    c_year  = pick(df, 'cycle', 'year')
    c_state = pick(df, 'state_abbrev', 'state')
    c_cand  = pick(df, 'candidate_name', 'candidate')
    c_party = pick(df, 'ballot_party', 'party')   # NOTE: 'party' is all-null in these files; ballot_party holds DEM/REP
    c_pct   = pick(df, 'percent', 'pct')
    c_win   = pick(df, 'winner')
    c_seat  = pick(df, 'office_seat_name', 'seat_name')
    out = pd.DataFrame({
        'year':      pd.to_numeric(df[c_year], errors='coerce'),
        'state':     df[c_state].map(to_abbr),          # standardize to abbrev (matches polls)
        'office':    office,
        'district':  (df[c_seat].map(parse_district) if (office == 'House' and c_seat) else ''),
        'res_candidate': df[c_cand],
        'res_party':     df[c_party].map(norm_party),
        'vote_pct':      pd.to_numeric(df[c_pct], errors='coerce'),
        'won':           df[c_win],
    })
    out['district'] = out['district'].fillna('')
    out['won'] = (out['won'].astype(str).str.lower()
                  .isin(['true','1','t','yes','y'])).astype(int)
    out['cand_key'] = out['res_candidate'].map(norm_name)
    out = out.dropna(subset=['year', 'cand_key'])
    out['year'] = out['year'].astype(int)
    # race_id includes district so each House district is its own race
    out['race_id'] = (out['year'].astype(str) + '_' + out['state'] + '_'
                      + out['office'] + out['district'].radd('-').where(out['district'] != '', ''))
    out['race_winning_pct'] = out.groupby('race_id')['vote_pct'].transform('max')
    return out

results = pd.concat([load_results(result_files[o], o) for o in OFFICES if result_files.get(o)],
                    ignore_index=True)
results = results[results['year'] >= MIN_YEAR]
res_join = results[['year','state','office','district','cand_key',
                    'won','vote_pct','res_party','res_candidate','race_winning_pct']]\
                  .drop_duplicates(['year','state','office','district','cand_key'])
print('result rows:', len(results), ' | unique candidate-races:', len(res_join))
print('res_party distribution:', results['res_party'].value_counts().to_dict())
res_join.head()

result rows: 20452  | unique candidate-races: 19353
res_party distribution: {'OTH': 8106, 'DEM': 6224, 'REP': 6122}


,year,state,office,district,cand_key,won,vote_pct,res_party,res_candidate,race_winning_pct
0,2024,VT,Senate,,schoville j,0,0.919194,OTH,Justin Schoville,63.159561
1,2024,VT,Senate,,hill m,0,1.247065,OTH,Matt Hill,63.159561
2,2024,VT,Senate,,berry s,0,2.186080,OTH,Steve Berry,63.159561
3,2024,VT,Senate,,malloy g,0,32.074615,REP,Gerald Malloy,63.159561
4,2024,VT,Senate,,sanders b,1,63.159561,OTH,Bernie Sanders,63.159561


## 5. Join: attach each race's result onto every poll row

Left join keeps **all** poll rows. `_merge`/`has_result` tells you which polls matched a result, so you keep maximum data and can filter later for modeling.

In [7]:
long = polls.merge(res_join, on=['year','state','office','district','cand_key'],
                   how='left', indicator=True)
long['has_result'] = (long['_merge'] == 'both').astype(int)
long = long.drop(columns='_merge')
# race_id includes district so each House district is its own race (Senate/Gov: district == '')
long['race_id'] = (long['year'].astype(str) + '_' + long['state'] + '_'
                   + long['office'] + long['district'].radd('-').where(long['district'] != '', ''))

print('long rows:', len(long))
print('match rate by office:')
print(long.groupby('office')['has_result'].mean().round(3))
print('\npolls with a known winner:', long['won'].notna().sum())
print('distinct races by office:')
print(long[long.has_result==1].groupby('office')['race_id'].nunique())
long[['year','state','office','district','candidate','pct',
      'res_candidate','won','vote_pct','has_result','race_id']].head(12)

long rows: 22546
match rate by office:
office
Governor    0.651
House       0.815
Senate      0.804
Name: has_result, dtype: float64

polls with a known winner: 17037
distinct races by office:
office
Governor     95
House       476
Senate      126
Name: race_id, dtype: int64


,year,state,office,district,candidate,pct,res_candidate,won,vote_pct,has_result,race_id
0,2026,ME,Senate,,Graham Platner,47.6,NaN,NaN,NaN,0,2026_ME_Senate
1,2026,ME,Senate,,Susan M. Collins,45.4,NaN,NaN,NaN,0,2026_ME_Senate
2,2026,GA,Senate,,Derek Dooley,48.0,NaN,NaN,NaN,0,2026_GA_Senate
3,2026,GA,Senate,,Mike Collins,50.0,NaN,NaN,NaN,0,2026_GA_Senate
4,2026,MA,Senate,,John Deaton,26.2,NaN,NaN,NaN,0,2026_MA_Senate
5,2026,MA,Senate,,Seth Moulton,53.8,NaN,NaN,NaN,0,2026_MA_Senate
6,2026,MA,Senate,,John Deaton,29.8,NaN,NaN,NaN,0,2026_MA_Senate
7,2026,MA,Senate,,Edward J. Markey,55.0,NaN,NaN,NaN,0,2026_MA_Senate
8,2026,GA,Senate,,Derek Dooley,46.0,NaN,NaN,NaN,0,2026_GA_Senate
9,2026,GA,Senate,,Mike Collins,48.0,NaN,NaN,NaN,0,2026_GA_Senate


## 6. Save the long dataset

This is the master file: one row per poll, result attached. Collapse to one-row-per-race later when you're ready to model.

In [8]:
OUT = 'polls_long_with_results.csv'
long.to_csv(OUT, index=False)
print('saved ->', os.path.abspath(OUT))
print('shape :', long.shape)
print('years :', long['year'].min(), '-', long['year'].max())
print('cols  :', list(long.columns))

saved -> C:\Users\pjmer\Documents\Polling prediction model\polls_long_with_results.csv
shape : (22546, 63)
years : 2018 - 2026
cols  : ['poll_id', 'pollster_id', 'pollster', 'sponsor_ids', 'sponsors', 'display_name', 'pollster_rating_id', 'pollster_rating_name', 'numeric_grade', 'pollscore', 'methodology', 'transparency_score', 'state', 'start_date', 'end_date', 'sponsor_candidate_id', 'sponsor_candidate', 'sponsor_candidate_party', 'question_id', 'sample_size', 'population', 'subpopulation', 'population_full', 'tracking', 'created_at', 'notes', 'url', 'url_article', 'url_topline', 'url_crosstab', 'source', 'internal', 'partisan', 'year', 'office_type', 'seat_name', 'seat_number', 'election_date', 'stage', 'poll_party', 'pct', 'answer', 'candidate', 'candidate_id', 'race_id', 'nationwide_match', 'ranked_choice_reallocated', 'hypothetical', 'office', 'district', 'cand_key', 'party_std', 'endorsed_candidate_id', 'endorsed_candidate_name', 'endorsed_candidate_party', 'nationwide_batch', '

## Notes

- **Long format on purpose:** every poll is a row, so you retain pollster, dates, sample size, methodology, ratings — aggregate however you like later (e.g. recency-weighted average, last-21-days, pollster-rating weighting).
- **`has_result == 0`** = a poll whose candidate didn't match a result row (usually a name-matching miss, a primary, or a dropped-out candidate). Keep them for now; filter to `has_result == 1` when training.
- **House** matches worst (sparse district polling); Senate/Governor match best.
- **Collapsing later:** group by `race_id`, build features per candidate, then pivot to one row per race. Don't feed `vote_pct` / `race_winning_pct` into the model as inputs — they're outcomes.
- If match rate looks low, inspect `long[long.has_result==0]` and refine `norm_name`.